# INJECTION-R4b: Domain-Boost + Joint Polish (Retry)

**Why R4 failed:** `best_epoch=0` — combined `val_loss` was dominated by cross-entropy noise,  
checkpoint never fired, and Non-IT class weight (0.8) was too low for the hardest class.

**Fix — two-phase approach:**
- **Phase 1 (Domain Boost):** Freeze encoder + ATS + RSG heads. Train domain head only with  
  heavily rebalanced class weights and higher LR. Target: push Domain F1 past 0.85.
- **Phase 2 (Joint Polish):** Unfreeze everything. Checkpoint on `val_dom_f1` (not combined CE loss).

### Final Hard Gates — ALL must pass
| Metric | Gate |
|--------|------|
| ATS val MAE (0–100) | < 6.5 |
| ATS test MAE (0–100) | < 7.0 |
| Domain F1 macro | > 0.85 |
| Domain F1 per-class (all 7) | > 0.80 each |
| RSG val Accuracy | > 0.65 |
| Fresher fairness gap | ≤ 20 pts |

### Required files — upload to `ATS_R4b/` on Google Drive
```
MyDrive/ATS_R4b/
  ats_tokenized.npz          <- from data/tokenized/
  rsg_tokenized.npz          <- from data/tokenized/
  data_splits.json           <- from data/tokenized/
  r4_final_weights.h5        <- from ATS_R4/output/ (== r3 baseline since best_epoch=0)
  merged_final.csv           <- from data/labeled/  (optional: fresher fairness)
```
> **Runtime:** T4 GPU — `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Cell 1: GPU check
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('No GPU! Go to Runtime -> Change runtime type -> T4 GPU then re-run.')
else:
    print(result.stdout)

In [ ]:
# Cell 2: Install dependencies
!pip install -q "transformers==4.44.2" scikit-learn
print('Done.')
print('>>> ACTION: Runtime -> Restart session -> re-run all cells from Cell 1 <<<')

In [ ]:
# Cell 3: Mount Drive & configure paths
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
#  CONFIGURE: your ATS_R4b folder on Drive
# ================================================================
DRIVE_ROOT = '/content/drive/MyDrive/ATS_R4b'

import os
from pathlib import Path

DATA_DIR     = Path(DRIVE_ROOT)
IN_WEIGHTS   = str(DATA_DIR / 'r4_final_weights.h5')   # input: R4 output (== R3 baseline)
SPLITS_JSON  = str(DATA_DIR / 'data_splits.json')
MERGED_CSV   = str(DATA_DIR / 'merged_final.csv')       # optional

OUT_DIR      = DATA_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)
P1_BEST      = str(OUT_DIR / 'r4b_phase1_best.h5')
P2_BEST      = str(OUT_DIR / 'r4b_phase2_ckpt.h5')
FINAL_W      = str(OUT_DIR / 'r4b_final_weights.h5')
FINAL_LOG    = str(OUT_DIR / 'r4b_training_log.csv')
FINAL_REPORT = str(OUT_DIR / 'r4b_final_eval_report.json')

print(f'Input weights : {IN_WEIGHTS}')
print(f'Output dir    : {OUT_DIR}')

In [ ]:
# Cell 4: Verify required files
required = [
    str(DATA_DIR / 'ats_tokenized.npz'),
    str(DATA_DIR / 'rsg_tokenized.npz'),
    IN_WEIGHTS,
    SPLITS_JSON,
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1e6:.1f} MB' if exists else 'MISSING'
    print(f'  [{"OK     " if exists else "MISSING"}]  {size:>10}   {f}')
    all_ok = all_ok and exists
exists_csv = os.path.exists(MERGED_CSV)
print(f'  [OPTIONAL]  {(str(round(os.path.getsize(MERGED_CSV)/1e6,1))+" MB") if exists_csv else "not uploaded":>10}   {MERGED_CSV}')
if not all_ok:
    raise FileNotFoundError('Missing required files — upload to Drive first.')
print('\nAll required files present.')

In [ ]:
# Cell 5: Imports & environment
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import csv, json
import numpy as np
import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
from sklearn.metrics import f1_score

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Cell 6: Hyperparameters — R4b two-phase config
MINILM_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SEQ_LEN           = 128

# ── Phase 1: Domain-only boost ──────────────────────────────────────────────
P1_LR         = 5e-5   # higher — only domain head params, safe to be aggressive
P1_BATCH      = 32
P1_MAX_EPOCHS = 15
P1_PATIENCE   = 5      # stop if val_dom_f1 doesn't improve
P1_TARGET_F1  = 0.85   # stop early once gate is hit

# Phase 1 domain class weights — aggressively boost failing classes
# R4 results: IT=0.774 Non-IT=0.742 Healthcare=0.790 Finance=0.765
#             Design=0.908 Legal=0.922 Edu=0.823
P1_DOM_WEIGHTS = {
    0: 2.2,   # IT        — was 1.4, failing at 0.774
    1: 3.0,   # Non-IT    — was 0.8 (WRONG!), worst class at 0.742
    2: 0.4,   # Design    — passing well at 0.908, reduce
    3: 2.2,   # Healthcare— was 1.0, failing at 0.790
    4: 2.5,   # Finance   — was 1.5, still failing at 0.765
    5: 0.3,   # Legal     — passing at 0.922, reduce
    6: 0.7,   # Edu       — passing at 0.823, slight reduction
}

# ── Phase 2: Joint polish ───────────────────────────────────────────────────
P2_LR         = 8e-6   # slightly higher than R4's 1e-5 to actually move weights
P2_BATCH      = 32
P2_MAX_EPOCHS = 30
P2_PATIENCE   = 8

# Canonical loss weights (spec)
W_ATS = 0.35
W_DOM = 0.35
W_RSG = 0.30

# Phase 2 domain class weights — balanced, still favour failing classes
P2_DOM_WEIGHTS = {
    0: 1.8,   # IT
    1: 2.2,   # Non-IT — still needs extra weight
    2: 0.5,   # Design
    3: 1.7,   # Healthcare
    4: 2.0,   # Finance
    5: 0.4,   # Legal
    6: 0.8,   # Edu
}

GRAD_CLIP  = 1.0      # gradient clipping to stabilise encoder fine-tuning
HARD_STOP  = 8.5      # abort if val ATS MAE exceeds this (0-100)

# Hard gate thresholds (spec)
GATE_ATS_VAL  = 6.5
GATE_ATS_TEST = 7.0
GATE_DOM_MAC  = 0.85
GATE_DOM_CLS  = 0.80
GATE_RSG      = 0.65
GATE_FRESH    = 20.0

DOMAIN_NAMES = ['IT', 'Non-IT', 'Design', 'Healthcare', 'Finance', 'Legal', 'Edu']
DOMAIN_HEAD_LAYERS = ['dom_dense1', 'dom_drop1', 'dom_dense2', 'dom_drop2', 'domain_probs']

FRESHER_KEYWORDS = [
    'fresher', 'entry level', 'entry-level', '0 years experience',
    '0-1 years', 'recent graduate', 'final year', 'fresh graduate',
    'no prior experience', 'no experience',
]

print('Hyperparameters configured.')
print(f'  Phase 1: lr={P1_LR}  max_epochs={P1_MAX_EPOCHS}  patience={P1_PATIENCE}')
print(f'  Phase 2: lr={P2_LR}  max_epochs={P2_MAX_EPOCHS}  patience={P2_PATIENCE}')
print(f'  Grad clip: {GRAD_CLIP}  |  ATS/DOM/RSG weights: {W_ATS}/{W_DOM}/{W_RSG}')

In [ ]:
# Cell 7: MiniLM model definition
import transformers as _tf_check
_major, _minor = (int(x) for x in _tf_check.__version__.split('.')[:2])
if (_major, _minor) >= (4, 47):
    raise RuntimeError(
        f'transformers {_tf_check.__version__} detected — TFAutoModel removed in 4.47.\n'
        'Run Cell 2, then Runtime -> Restart session -> re-run from Cell 1.'
    )

from transformers import TFAutoModel


def mean_pool_l2(last_hidden, attention_mask):
    mask    = tf.cast(tf.expand_dims(attention_mask, -1), tf.float32)
    sum_emb = tf.reduce_sum(last_hidden * mask, axis=1)
    count   = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-9)
    return tf.nn.l2_normalize(sum_emb / count, axis=-1)


class MiniLMEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model, **kwargs):
        super().__init__(**kwargs)
        self._bert = bert_model

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        out = self._bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=tf.zeros_like(input_ids),
            training=training,
        )
        return mean_pool_l2(out.last_hidden_state, attention_mask)

    def get_config(self):
        return super().get_config()


def build_unified_minilm_model(bert_model):
    resume_ids  = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_input_ids')
    resume_mask = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_attention_mask')
    jd_ids      = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_input_ids')
    jd_mask     = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_attention_mask')

    encoder    = MiniLMEncoderLayer(bert_model, name='minilm_encoder')
    resume_emb = encoder([resume_ids,  resume_mask])
    jd_emb     = encoder([jd_ids,      jd_mask])

    cosine_sim = tf.keras.layers.Dot(axes=1, normalize=True,  name='cosine_sim')([resume_emb, jd_emb])
    dot_prod   = tf.keras.layers.Dot(axes=1, normalize=False, name='dot_prod')  ([resume_emb, jd_emb])
    ats_feat   = tf.keras.layers.Concatenate(name='ats_features')([resume_emb, jd_emb, cosine_sim, dot_prod])

    x1        = tf.keras.layers.Dense(256, activation='relu',   name='ats_dense1')(ats_feat)
    x1        = tf.keras.layers.Dropout(0.3,                    name='ats_drop1')(x1)
    x1        = tf.keras.layers.Dense(64,  activation='relu',   name='ats_dense2')(x1)
    x1        = tf.keras.layers.Dropout(0.2,                    name='ats_drop2')(x1)
    ats_score = tf.keras.layers.Dense(1, activation='sigmoid',  name='ats_score')(x1)

    x2           = tf.keras.layers.Dense(256, activation='relu',  name='dom_dense1')(jd_emb)
    x2           = tf.keras.layers.Dropout(0.3,                   name='dom_drop1')(x2)
    x2           = tf.keras.layers.Dense(128, activation='relu',  name='dom_dense2')(x2)
    x2           = tf.keras.layers.Dropout(0.2,                   name='dom_drop2')(x2)
    domain_probs = tf.keras.layers.Dense(7, activation='softmax', name='domain_probs')(x2)

    x3           = tf.keras.layers.Dense(512, activation='relu',  name='rsg_dense1')(resume_emb)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn1')(x3)
    x3           = tf.keras.layers.Dropout(0.4,                   name='rsg_drop1')(x3)
    x3           = tf.keras.layers.Dense(256, activation='relu',  name='rsg_dense2')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn2')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop2')(x3)
    x3           = tf.keras.layers.Dense(128, activation='relu',  name='rsg_dense3')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn3')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop3')(x3)
    rsg_template = tf.keras.layers.Dense(46, activation='softmax', name='rsg_template')(x3)

    return tf.keras.Model(
        inputs=[resume_ids, resume_mask, jd_ids, jd_mask],
        outputs=[ats_score, domain_probs, rsg_template],
        name='unified_ats_minilm',
    )

print(f'transformers {_tf_check.__version__} OK.')
print('Model definition ready.')

In [ ]:
# Cell 8: Load tokenized data & splits
print('Loading tokenized data...')
ats_data = np.load(str(DATA_DIR / 'ats_tokenized.npz'))
rsg_data = np.load(str(DATA_DIR / 'rsg_tokenized.npz'))

with open(SPLITS_JSON) as f:
    splits = json.load(f)

ats_tr_idx  = np.array(splits['ats_train'])
ats_vl_idx  = np.array(splits['ats_val'])
ats_tst_idx = np.array(splits['ats_test'])
rsg_tr_idx  = np.array(splits['rsg_train'])
rsg_vl_idx  = np.array(splits['rsg_val'])

ATS_KEYS = ('resume_input_ids', 'resume_attention_mask',
             'jd_input_ids', 'jd_attention_mask', 'ats_scores', 'domain_labels')
RSG_KEYS = ('profile_input_ids', 'profile_attention_mask', 'rsg_labels')

ats_tr  = {k: ats_data[k][ats_tr_idx]  for k in ATS_KEYS}
ats_vl  = {k: ats_data[k][ats_vl_idx]  for k in ATS_KEYS}
ats_tst = {k: ats_data[k][ats_tst_idx] for k in ATS_KEYS}
rsg_tr  = {k: rsg_data[k][rsg_tr_idx]  for k in RSG_KEYS}
rsg_vl  = {k: rsg_data[k][rsg_vl_idx]  for k in RSG_KEYS}

n_ats_tr = len(ats_tr_idx)
n_rsg_tr = len(rsg_tr_idx)
print(f'  ATS  train={n_ats_tr:,}  val={len(ats_vl_idx):,}  test={len(ats_tst_idx):,}')
print(f'  RSG  train={n_rsg_tr:,}  val={len(rsg_vl_idx):,}')

In [ ]:
# Cell 9: Build model & load input weights
print(f'Loading MiniLM ({MINILM_MODEL_NAME})...')
bert_model = TFAutoModel.from_pretrained(MINILM_MODEL_NAME, from_pt=True)
bert_model.trainable = True

model = build_unified_minilm_model(bert_model)
model.load_weights(IN_WEIGHTS)
print(f'Weights loaded: {IN_WEIGHTS}')
print(f'Total params  : {model.count_params():,}')

In [ ]:
# Cell 10: Eval helpers (shared across both phases)
_sce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def eval_ats(data, batch=64):
    """Returns (mae_100, dom_f1_macro, per_cls_f1, dom_ce, mae_01)."""
    n = len(data['ats_scores'])
    ats_p, dom_p = [], []
    for s in range(0, n, batch):
        e = min(s + batch, n)
        ap, dp, _ = model(
            [data['resume_input_ids'][s:e], data['resume_attention_mask'][s:e],
             data['jd_input_ids'][s:e],     data['jd_attention_mask'][s:e]],
            training=False,
        )
        ats_p.append(ap.numpy())
        dom_p.append(dp.numpy())
    ats_p = np.concatenate(ats_p).squeeze(-1)
    dom_p = np.concatenate(dom_p)

    mae_01       = float(np.mean(np.abs(ats_p - data['ats_scores'])))
    mae_100      = mae_01 * 100.0
    dom_true     = data['domain_labels']
    dom_pred_cls = np.argmax(dom_p, 1)
    dom_f1_macro = f1_score(dom_true, dom_pred_cls, average='macro',  zero_division=0)
    per_cls_f1   = f1_score(dom_true, dom_pred_cls, average=None,
                            labels=list(range(7)), zero_division=0)
    dom_ce       = float(_sce(
        dom_true.astype('int32'), dom_p
    ).numpy().mean())
    return mae_100, dom_f1_macro, per_cls_f1, dom_ce, mae_01


def eval_rsg(data, batch=64):
    """Returns (top1_acc, top3_acc, ce)."""
    m = len(data['rsg_labels'])
    rsg_p = []
    for s in range(0, m, batch):
        e    = min(s + batch, m)
        pids = data['profile_input_ids'][s:e]
        pmsk = data['profile_attention_mask'][s:e]
        _, _, rp = model([pids, pmsk, pids, pmsk], training=False)
        rsg_p.append(rp.numpy())
    rsg_p    = np.concatenate(rsg_p)
    labels   = data['rsg_labels']
    top1_acc = float(np.mean(np.argmax(rsg_p, 1) == labels))
    top3_preds = np.argsort(rsg_p, axis=1)[:, -3:]
    top3_acc = float(np.mean([labels[i] in top3_preds[i] for i in range(m)]))
    rsg_ce   = float(_sce(labels.astype('int32'), rsg_p).numpy().mean())
    return top1_acc, top3_acc, rsg_ce


print('Eval helpers ready.')

In [ ]:
# Cell 11: Baseline — R4 output metrics (starting point for R4b)
print('Measuring input-weights baseline (val set)...')
b_mae, b_dom_f1, b_per_cls, _, _ = eval_ats(ats_vl)
b_rsg_acc, b_rsg_top3, _         = eval_rsg(rsg_vl)

print(f'  val ATS MAE  : {b_mae:.2f}   (gate < {GATE_ATS_VAL})')
print(f'  val Dom F1   : {b_dom_f1:.4f}  (gate > {GATE_DOM_MAC})')
print(f'  val RSG Top-1: {b_rsg_acc:.4f}  (gate > {GATE_RSG})')
print(f'  val RSG Top-3: {b_rsg_top3:.4f}')
print()
print('  Per-domain F1 (baseline):')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, b_per_cls)):
    flag = 'PASS' if f1v > GATE_DOM_CLS else 'FAIL <-- must fix'
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}')

In [ ]:
# Cell 12: ── PHASE 1: Domain-only boost ──────────────────────────────────────
print('=' * 65)
print('  PHASE 1: Domain-only boost')
print(f'  Frozen: encoder + ATS head + RSG head')
print(f'  Trainable: domain head only ({DOMAIN_HEAD_LAYERS})')
print(f'  LR={P1_LR}  max_epochs={P1_MAX_EPOCHS}  patience={P1_PATIENCE}')
print('=' * 65)

# Freeze everything, then unfreeze domain head only
for layer in model.layers:
    layer.trainable = False
bert_model.trainable = False

for name in DOMAIN_HEAD_LAYERS:
    try:
        model.get_layer(name).trainable = True
    except ValueError:
        print(f'  WARNING: layer {name!r} not found — check model definition.')

trainable_params = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
print(f'\n  Trainable params: {trainable_params:,} (domain head only)')

# Pre-compute Phase 1 sample weights
p1_dom_w = np.array(
    [P1_DOM_WEIGHTS.get(int(d), 1.0) for d in ats_tr['domain_labels']],
    dtype='float32'
)

opt_p1  = tf.keras.optimizers.Adam(learning_rate=P1_LR, clipnorm=GRAD_CLIP)
_sce_p1 = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


@tf.function
def train_step_dom(r_ids, r_mask, jd_ids, jd_mask, dom_y, dom_w):
    with tf.GradientTape() as tape:
        _, dom_p, _ = model([r_ids, r_mask, jd_ids, jd_mask], training=True)
        loss = _sce_p1(dom_y, dom_p, sample_weight=dom_w)
    opt_p1.apply_gradients(
        zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables)
    )
    return loss


rng            = np.random.default_rng(42)
best_p1_f1     = b_dom_f1
p1_patience    = 0
p1_log         = []
n_ats_batches  = -(-n_ats_tr // P1_BATCH)

print(f'  ATS batches/epoch: {n_ats_batches}\n')

for epoch in range(1, P1_MAX_EPOCHS + 1):
    perm      = rng.permutation(n_ats_tr)
    dom_ls    = []

    for bi in range(n_ats_batches):
        idx = perm[bi * P1_BATCH : (bi + 1) * P1_BATCH]
        l = train_step_dom(
            ats_tr['resume_input_ids'][idx],
            ats_tr['resume_attention_mask'][idx],
            ats_tr['jd_input_ids'][idx],
            ats_tr['jd_attention_mask'][idx],
            ats_tr['domain_labels'][idx].astype('int32'),
            p1_dom_w[idx],
        )
        dom_ls.append(float(l))

    if np.isnan(float(np.mean(dom_ls))):
        print(f'HARD STOP — NaN at epoch {epoch}')
        break

    _, val_dom_f1, val_per_cls, _, val_mae_01 = eval_ats(ats_vl)
    val_mae_100 = val_mae_01 * 100.0

    print(
        f'  P1 Ep {epoch:02d}/{P1_MAX_EPOCHS} | '
        f'dom_loss:{np.mean(dom_ls):.4f}  '
        f'val_DomF1:{val_dom_f1:.4f}  '
        f'val_MAE:{val_mae_100:.2f}  '
        f'worst_cls:{min(val_per_cls):.4f}'
    )

    p1_log.append({
        'phase': 1, 'epoch': epoch,
        'dom_loss':    round(float(np.mean(dom_ls)), 6),
        'val_dom_f1':  round(val_dom_f1, 4),
        'val_mae_100': round(val_mae_100, 4),
        'worst_cls_f1': round(float(min(val_per_cls)), 4),
    })

    if val_dom_f1 > best_p1_f1:
        best_p1_f1  = val_dom_f1
        p1_patience = 0
        model.save_weights(P1_BEST)
        print(f'    -> New best DomF1={val_dom_f1:.4f} -- saved.')
    else:
        p1_patience += 1
        print(f'    -> No improvement ({p1_patience}/{P1_PATIENCE})')
        if p1_patience >= P1_PATIENCE:
            print(f'\n  Phase 1 early stop at epoch {epoch}.')
            break

    # Early exit once gate is already cleared
    if val_dom_f1 >= P1_TARGET_F1 and all(f > GATE_DOM_CLS for f in val_per_cls):
        print(f'\n  Phase 1 gate achieved at epoch {epoch}! Proceeding to Phase 2.')
        break

# Restore best Phase 1 weights
if os.path.exists(P1_BEST):
    model.load_weights(P1_BEST)
    print(f'\nPhase 1 complete. Best DomF1: {best_p1_f1:.4f}')
else:
    print('\nPhase 1: no improvement over baseline — continuing with input weights.')

In [ ]:
# Cell 13: Phase 1 results summary
print('Phase 1 end-state (val):')
p1_mae, p1_dom_f1, p1_per_cls, _, _ = eval_ats(ats_vl)
p1_rsg_acc, p1_rsg_top3, _          = eval_rsg(rsg_vl)

print(f'  val ATS MAE  : {p1_mae:.2f}   baseline was {b_mae:.2f}')
print(f'  val Dom F1   : {p1_dom_f1:.4f}  baseline was {b_dom_f1:.4f}  delta={p1_dom_f1-b_dom_f1:+.4f}')
print(f'  val RSG Top-1: {p1_rsg_acc:.4f}  baseline was {b_rsg_acc:.4f}')
print()
print('  Per-domain F1 after Phase 1:')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, p1_per_cls)):
    delta = f1v - b_per_cls[i]
    flag  = 'PASS' if f1v > GATE_DOM_CLS else 'FAIL'
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}  (delta {delta:+.4f})')

In [ ]:
# Cell 14: ── PHASE 2: Joint polish (unfreeze everything) ─────────────────────
print('=' * 65)
print('  PHASE 2: Joint polish — encoder + all heads unfrozen')
print(f'  LR={P2_LR}  clipnorm={GRAD_CLIP}')
print(f'  Checkpoint metric: val_dom_f1 (primary gate)')
print('=' * 65)

# Unfreeze everything
for layer in model.layers:
    layer.trainable = True
bert_model.trainable = True

trainable_p2 = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
print(f'  Trainable params: {trainable_p2:,} (all layers)\n')

# Pre-compute Phase 2 sample weights
p2_dom_w = np.array(
    [P2_DOM_WEIGHTS.get(int(d), 1.0) for d in ats_tr['domain_labels']],
    dtype='float32'
)

opt_p2   = tf.keras.optimizers.Adam(learning_rate=P2_LR, clipnorm=GRAD_CLIP)
_mae_fn  = tf.keras.losses.MeanAbsoluteError()
_sce_p2  = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
_sce_rsg = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def train_step_ats_dom(r_ids, r_mask, jd_ids, jd_mask, ats_y, dom_y, dom_w):
    with tf.GradientTape() as tape:
        ats_p, dom_p, _ = model([r_ids, r_mask, jd_ids, jd_mask], training=True)
        ats_l = _mae_fn(tf.expand_dims(ats_y, 1), ats_p)
        dom_l = _sce_p2(dom_y, dom_p, sample_weight=dom_w)
        total = W_ATS * ats_l + W_DOM * dom_l
    opt_p2.apply_gradients(
        zip(tape.gradient(total, model.trainable_variables), model.trainable_variables)
    )
    return float(ats_l), float(dom_l)


def train_step_rsg(p_ids, p_mask, rsg_y):
    with tf.GradientTape() as tape:
        _, _, rsg_p = model([p_ids, p_mask, p_ids, p_mask], training=True)
        loss = W_RSG * _sce_rsg(rsg_y, rsg_p)
    opt_p2.apply_gradients(
        zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables)
    )
    return float(loss)


best_p2_dom_f1 = p1_dom_f1     # checkpoint when domain F1 improves
best_p2_epoch  = 0
p2_patience    = 0
p2_log         = []
hard_stop      = False
n_ats_batches  = -(-n_ats_tr // P2_BATCH)

# Save Phase 1 best as Phase 2 starting checkpoint
model.save_weights(P2_BEST)

print(f'  ATS batches/epoch: {n_ats_batches}  |  RSG train: {n_rsg_tr:,}\n')

for epoch in range(1, P2_MAX_EPOCHS + 1):
    ats_perm = rng.permutation(n_ats_tr)
    rsg_perm = rng.permutation(n_rsg_tr)
    rsg_pool = np.tile(rsg_perm, (n_ats_batches // max(1, n_rsg_tr // P2_BATCH) + 2))
    rsg_cur  = 0

    ats_ls, dom_ls, rsg_ls = [], [], []

    for bi in range(n_ats_batches):
        idx_a = ats_perm[bi * P2_BATCH : (bi + 1) * P2_BATCH]
        al, dl = train_step_ats_dom(
            ats_tr['resume_input_ids'][idx_a],
            ats_tr['resume_attention_mask'][idx_a],
            ats_tr['jd_input_ids'][idx_a],
            ats_tr['jd_attention_mask'][idx_a],
            ats_tr['ats_scores'][idx_a].astype('float32'),
            ats_tr['domain_labels'][idx_a].astype('int32'),
            p2_dom_w[idx_a],
        )
        ats_ls.append(al)
        dom_ls.append(dl)

        # Interleaved RSG batch
        idx_r = rsg_pool[rsg_cur * P2_BATCH : (rsg_cur + 1) * P2_BATCH]
        idx_r = idx_r[:P2_BATCH]
        rsg_cur = (rsg_cur + 1) % max(1, n_rsg_tr // P2_BATCH)
        rl = train_step_rsg(
            rsg_tr['profile_input_ids'][idx_r],
            rsg_tr['profile_attention_mask'][idx_r],
            rsg_tr['rsg_labels'][idx_r].astype('int32'),
        )
        rsg_ls.append(rl)

    if np.isnan(float(np.mean(ats_ls))):
        print(f'\nHARD STOP — NaN loss at epoch {epoch}')
        model.load_weights(P2_BEST)
        hard_stop = True
        break

    # Validation
    val_mae, val_dom_f1, val_per_cls, _, _ = eval_ats(ats_vl)
    val_rsg_acc, val_rsg_top3, _           = eval_rsg(rsg_vl)

    print(
        f'  P2 Ep {epoch:02d}/{P2_MAX_EPOCHS} | '
        f'tr_ATS:{np.mean(ats_ls):.4f}  '
        f'val_MAE:{val_mae:.2f}  '
        f'val_DomF1:{val_dom_f1:.4f}  '
        f'val_RSG:{val_rsg_acc:.3f}  '
        f'worst_cls:{min(val_per_cls):.4f}'
    )

    p2_log.append({
        'phase': 2, 'epoch': epoch,
        'train_ats_mae':  round(float(np.mean(ats_ls)), 6),
        'train_dom_loss': round(float(np.mean(dom_ls)), 6),
        'train_rsg_loss': round(float(np.mean(rsg_ls)), 6),
        'val_mae_100':    round(val_mae, 4),
        'val_dom_f1':     round(val_dom_f1, 4),
        'val_rsg_acc':    round(val_rsg_acc, 4),
        'worst_cls_f1':   round(float(min(val_per_cls)), 4),
    })

    # ATS regression hard stop
    if val_mae > HARD_STOP:
        print(f'\n  *** HARD STOP epoch {epoch}: val MAE {val_mae:.2f} > {HARD_STOP}. Restoring best. ***')
        model.load_weights(P2_BEST)
        hard_stop = True
        break

    # Checkpoint on val_dom_f1 (the critical failing gate)
    if val_dom_f1 > best_p2_dom_f1:
        best_p2_dom_f1 = val_dom_f1
        best_p2_epoch  = epoch
        p2_patience    = 0
        model.save_weights(P2_BEST)
        print(f'    -> New best DomF1={val_dom_f1:.4f} (MAE={val_mae:.2f}) -- checkpoint saved.')
    else:
        p2_patience += 1
        print(f'    -> No improvement ({p2_patience}/{P2_PATIENCE})')
        if p2_patience >= P2_PATIENCE:
            print(f'\n  Phase 2 early stop at epoch {epoch}.')
            model.load_weights(P2_BEST)
            break

print(f'\nPhase 2 complete. Best DomF1: {best_p2_dom_f1:.4f} at epoch {best_p2_epoch}')

In [ ]:
# Cell 15: Save training log (both phases)
all_log = p1_log + p2_log
if all_log:
    with open(FINAL_LOG, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=all_log[0].keys())
        writer.writeheader()
        writer.writerows(all_log)
    print(f'Training log saved -> {FINAL_LOG}')

In [ ]:
# Cell 16: Save final weights
model.load_weights(P2_BEST)   # ensure best checkpoint is active
model.save_weights(FINAL_W)
print(f'Final weights saved -> {FINAL_W}')

In [ ]:
# Cell 17: Full test + val evaluation
print('Running full evaluation...')

test_mae, test_dom_f1, test_per_cls, _, _ = eval_ats(ats_tst)
val_mae_f, val_dom_f1_f, val_per_cls_f, _, _ = eval_ats(ats_vl)
rsg_top1, rsg_top3, _ = eval_rsg(rsg_vl)

print(f'  ATS val  MAE : {val_mae_f:.2f}')
print(f'  ATS test MAE : {test_mae:.2f}')
print(f'  Dom test F1  : {test_dom_f1:.4f}')
print(f'  RSG val Top-1: {rsg_top1:.4f}')
print(f'  RSG val Top-3: {rsg_top3:.4f}')

In [ ]:
# Cell 18: Fresher fairness check
fresher_gap   = None
n_fresher     = 0
n_experienced = 0
fresher_status = 'SKIPPED'

if os.path.exists(MERGED_CSV):
    print('Loading merged_final.csv...')
    import pandas as pd
    df = pd.read_csv(MERGED_CSV)
    if len(df) > max(ats_tst_idx):
        test_resumes = df.iloc[ats_tst_idx]['resume_text'].fillna('').values
        is_fresher   = np.array(
            [any(kw in r.lower() for kw in FRESHER_KEYWORDS) for r in test_resumes]
        )
        n_fresher    = int(is_fresher.sum())
        n_experienced = len(is_fresher) - n_fresher
        if n_fresher >= 10:
            all_scores = []
            n_tst = len(ats_tst_idx)
            for s in range(0, n_tst, 64):
                e = min(s + 64, n_tst)
                ap, _, _ = model(
                    [ats_tst['resume_input_ids'][s:e],
                     ats_tst['resume_attention_mask'][s:e],
                     ats_tst['jd_input_ids'][s:e],
                     ats_tst['jd_attention_mask'][s:e]], training=False)
                all_scores.append(ap.numpy().flatten())
            all_scores  = np.concatenate(all_scores) * 100.0
            fresher_gap = float(all_scores[~is_fresher].mean()) - float(all_scores[is_fresher].mean())
            fresher_status = 'PASS' if fresher_gap <= GATE_FRESH else 'FAIL'
            print(f'  Fresher: {n_fresher}  Experienced: {n_experienced}')
            print(f'  Gap    : {fresher_gap:.1f} pts  {fresher_status}')
        else:
            fresher_status = f'N/A (only {n_fresher} fresher samples)'
            print(f'  {fresher_status}')
    else:
        fresher_status = 'SKIPPED (index mismatch)'
        print(f'  {fresher_status}')
else:
    print('  merged_final.csv not found — fresher gate skipped.')

In [ ]:
# Cell 19: Gate report
g_ats_val  = val_mae_f       < GATE_ATS_VAL
g_ats_test = test_mae        < GATE_ATS_TEST
g_dom_mac  = test_dom_f1     > GATE_DOM_MAC
g_dom_cls  = all(f > GATE_DOM_CLS for f in test_per_cls)
g_rsg      = rsg_top1        > GATE_RSG
g_fresh    = (fresher_gap is None) or (fresher_gap <= GATE_FRESH)
all_pass   = g_ats_val and g_ats_test and g_dom_mac and g_dom_cls and g_rsg and g_fresh

print('\n' + '=' * 65)
print('  INJECTION-R4b — FINAL GATE RESULTS')
print('=' * 65)
print(f'  ATS val  MAE (0-100): {val_mae_f:.2f}   {"PASS" if g_ats_val  else "FAIL"}  (gate < {GATE_ATS_VAL})')
print(f'  ATS test MAE (0-100): {test_mae:.2f}   {"PASS" if g_ats_test else "FAIL"}  (gate < {GATE_ATS_TEST})')
print(f'  Domain F1 macro     : {test_dom_f1:.4f}  {"PASS" if g_dom_mac  else "FAIL"}  (gate > {GATE_DOM_MAC})')
print(f'  RSG val Top-1 Acc   : {rsg_top1:.4f}  {"PASS" if g_rsg      else "FAIL"}  (gate > {GATE_RSG})')
print(f'  RSG val Top-3 Acc   : {rsg_top3:.4f}')
print()
print(f'  Per-domain F1 (gate > {GATE_DOM_CLS} each):')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, test_per_cls)):
    delta = f1v - b_per_cls[i]
    flag  = 'PASS' if f1v > GATE_DOM_CLS else 'FAIL'
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}  (delta from R4: {delta:+.4f})')
print()
if fresher_gap is not None:
    print(f'  Fresher gap : {fresher_gap:.1f} pts  {"PASS" if g_fresh else "FAIL"}  (gate <= {GATE_FRESH})')
    print(f'    Fresher: {n_fresher}  |  Experienced: {n_experienced}')
else:
    print(f'  Fresher gap : {fresher_status}')
print()
print(f'  Phase 1 best DomF1  : {best_p1_f1:.4f}')
print(f'  Phase 2 best DomF1  : {best_p2_dom_f1:.4f} (epoch {best_p2_epoch})')
print(f'  Hard stop triggered : {"YES" if hard_stop else "NO"}')
print(f'  Output              : {FINAL_W}')
print('=' * 65)

if all_pass:
    print('\nR4b PASSED ALL GATES — proceed to T1')
else:
    print('\n*** R4b FAILED one or more gates — DO NOT proceed to T1. ***')

In [ ]:
# Cell 20: Save r4b_final_eval_report.json
report = {
    'stage': 'R4b',
    'all_gates_pass': bool(all_pass),
    'hard_stop': hard_stop,
    'phase1': {'best_dom_f1': round(best_p1_f1, 4)},
    'phase2': {'best_dom_f1': round(best_p2_dom_f1, 4), 'best_epoch': best_p2_epoch},
    'gates': {
        'ats_val_mae_100':  {'value': round(val_mae_f, 4),   'threshold': f'< {GATE_ATS_VAL}',  'pass': bool(g_ats_val)},
        'ats_test_mae_100': {'value': round(test_mae, 4),    'threshold': f'< {GATE_ATS_TEST}', 'pass': bool(g_ats_test)},
        'domain_f1_macro':  {'value': round(test_dom_f1, 4), 'threshold': f'> {GATE_DOM_MAC}',  'pass': bool(g_dom_mac)},
        'domain_all_cls_pass': bool(g_dom_cls),
        'rsg_val_top1':     {'value': round(rsg_top1, 4),    'threshold': f'> {GATE_RSG}',      'pass': bool(g_rsg)},
        'rsg_val_top3':     round(rsg_top3, 4),
        'fresher_gap':      fresher_gap if fresher_gap is not None else fresher_status,
        'fresher_pass':     bool(g_fresh),
    },
    'per_domain_f1': {
        DOMAIN_NAMES[i]: {
            'f1': round(float(test_per_cls[i]), 4),
            'pass': bool(test_per_cls[i] > GATE_DOM_CLS),
            'delta_from_r4': round(float(test_per_cls[i] - b_per_cls[i]), 4),
        }
        for i in range(7)
    },
    'baseline_r4': {
        'val_mae': round(b_mae, 4),
        'dom_f1':  round(b_dom_f1, 4),
        'rsg_acc': round(b_rsg_acc, 4),
    },
    'training_config': {
        'phase1': {'lr': P1_LR, 'max_epochs': P1_MAX_EPOCHS, 'patience': P1_PATIENCE,
                   'dom_weights': P1_DOM_WEIGHTS},
        'phase2': {'lr': P2_LR, 'max_epochs': P2_MAX_EPOCHS, 'patience': P2_PATIENCE,
                   'loss_weights': {'ats': W_ATS, 'dom': W_DOM, 'rsg': W_RSG},
                   'dom_weights': P2_DOM_WEIGHTS, 'grad_clip': GRAD_CLIP},
    },
}

with open(FINAL_REPORT, 'w') as fh:
    json.dump(report, fh, indent=2)
print(f'Report saved -> {FINAL_REPORT}')

In [ ]:
# Cell 21: Download results
from google.colab import files

print('Downloading r4b_final_weights.h5 ...')
files.download(FINAL_W)

print('Downloading r4b_final_eval_report.json ...')
files.download(FINAL_REPORT)

print('Downloading r4b_training_log.csv ...')
files.download(FINAL_LOG)

print('\nSave to local project:')
print('  r4b_final_weights.h5      -> model/unified_model/r4b_final_weights.h5')
print('  r4b_final_eval_report.json -> evaluation/r4b_final_eval_report.json')